In [1]:
import numpy as np
import pandas as pd

In [ ]:
import numpy as np
import pandas as pd

# --- Fantasy Football League Settings and Scoring ---
class LeagueSettings:
    def __init__(self, half_ppr=0.5, pass_yd_pts=0.04, pass_td_pts=4,
                 pass_int_pts=-1, rush_yd_pts=0.1, rush_td_pts=6,
                 rec_yd_pts=0.1, rec_td_pts=6, fumble_lost_pts=-2,
                 two_pt_conv_pts=2, sack_pts=3, int_pts=5,
                 fumble_rec_pts=2, idp_td_pts=6, safety_pts=2,
                 rushing_1st_down=0.5, rec_1st_down=0.5,
                 fumble=-1, fum_rec_td=6,
                 tfl=1, blocked_punt_pat_fg=2, ass_tak=0.5
                 forced_fumble=3, solo_tackle= 1,
                 pass_defended=2):
        """
        Initializes the fantasy football league scoring settings.
        """
        self.ppr = half_ppr
        self.pass_yd_pts = pass_yd_pts
        self.pass_td_pts = pass_td_pts
        self.pass_int_pts = pass_int_pts
        self.rush_yd_pts = rush_yd_pts
        self.rush_td_pts = rush_td_pts
        self.rec_yd_pts = rec_yd_pts
        self.rec_td_pts = rec_td_pts
        self.fumble_lost_pts = fumble_lost_pts
        self.two_pt_conv_pts = two_pt_conv_pts
        self.sack_pts = sack_pts
        self.int_pts = int_pts
        self.fumble_rec_pts = fumble_rec_pts
        self.def_td_pts = def_td_pts
        self.safety_pts = safety_pts
        self.idp_td_pts = idp_td_pts
        self.rush_1st_down = rushing_1st_down
        self.rec_1st_down = rec_1st_down
        self.fumble = fumble
        self.fum_rec_td = fum_rec_td
        self.tfl = tfl
        self.blocked_punt_pat_fg = blocked_punt_pat_fg
        self.ass_tak = ass_tak
        self.forced_fumble = forced_fumble
        self.solo_tackle = solo_tackle
        self.pass_defended = pass_defended

In [ ]:
class Player:
    def __init__(self, name, position, projections):
        """
        Represents a player with their projected stats.

        Args:
            name (str): Player's name.
            position (str): Player's position (QB, RB, WR, TE, K, DEF).
            projections (dict): Dictionary of projected stats.
        """
        self.name = name
        self.position = position
        self.projections = projections

    def get_projected_value(self, stat):
        """
        Returns a player's projected value for a given stat.
        Returns 0 if the stat is not found.
        """
        return self.projections.get(stat, 0)

In [ ]:
class FantasyCalculator:
    def __init__(self, league_settings):
        """
        Calculates fantasy points based on league settings and player projections.
        """
        self.settings = league_settings

    def calculate_fantasy_points(self, player):
        """
        Calculates the total fantasy points for a player.

        Args:
            player (Player): The player object.

        Returns:
            float: The total fantasy points.
        """
        points = 0
        proj = player.projections

        if player.position in ["QB", "RB", "WR", "TE"]:
            points += proj.get("passing_yds", 0) * self.settings.pass_yd_pts
            points += proj.get("passing_tds", 0) * self.settings.pass_td_pts
            points += proj.get("passing_ints", 0) * self.settings.pass_int_pts
            points += proj.get("rushing_yds", 0) * self.settings.rush_yd_pts
            points += proj.get("rushing_tds", 0) * self.settings.rush_td_pts
            points += proj.get("receptions", 0) * self.settings.ppr
            points += proj.get("receiving_yds", 0) * self.settings.rec_yd_pts
            points += proj.get("receiving_tds", 0) * self.settings.rec_td_pts
            points += proj.get("fumbles_lost", 0) * self.settings.fumble_lost_pts
            points += proj.get("two_pt_conversions", 0) * self.settings.two_pt_conv_pts
        elif player.position == "K":
            points += proj.get("fg_0_39", 0) * self.settings.fg_0_39_pts
            points += proj.get("fg_40_49", 0) * self.settings.fg_40_49_pts
            points += proj.get("fg_50_plus", 0) * self.settings.fg_50_plus_pts
            points += proj.get("fg_missed", 0) * self.settings.fg_missed_pts
            points += proj.get("xp_made", 0) * self.settings.xp_pts
            points += proj.get("xp_missed", 0) * self.settings.xp_missed_pts
        elif player.position == "DEF":
            points += proj.get("sacks", 0) * self.settings.sack_pts
            points += proj.get("interceptions", 0) * self.settings.int_pts
            points += proj.get("fumbles_recovered", 0) * self.settings.fumble_rec_pts
            points += proj.get("defensive_tds", 0) * self.settings.def_td_pts
            points += proj.get("safeties", 0) * self.settings.safety_pts

            # Points allowed scoring
            points_allowed = proj.get("points_allowed", 0)
            if points_allowed == 0:
                points += self.settings.pts_allowed_0_pts
            elif 1 <= points_allowed <= 6:
                points += self.settings.pts_allowed_1_6_pts
            elif 7 <= points_allowed <= 13:
                points += self.settings.pts_allowed_7_13_pts
            elif 14 <= points_allowed <= 20:
                points += self.settings.pts_allowed_14_20_pts
            elif 21 <= points_allowed <= 27:
                points += self.settings.pts_allowed_21_27_pts
            elif 28 <= points_allowed <= 34:
                points += self.settings.pts_allowed_28_34_pts
            else:
                points += self.settings.pts_allowed_35_plus_pts
            
            # Yards allowed scoring
            yards_allowed = proj.get("yards_allowed", 0)
            if yards_allowed < self.settings.yds_allowed_thresholds[0]:
                points += self.settings.yds_allowed_points[0]
            elif yards_allowed < self.settings.yds_allowed_thresholds[1]:
                points += self.settings.yds_allowed_points[1]
            elif yards_allowed < self.settings.yds_allowed_thresholds[2]:
                points += self.settings.yds_allowed_points[2]
            else:
                points += self.settings.yds_allowed_points[3]

        return points

In [ ]:
class JacobianCalculator:
    def __init__(self, league_settings, players):
        """
        Calculates the Jacobian matrix of fantasy point sensitivities.
        """
        self.settings = league_settings
        self.players = players
        self.calculator = FantasyCalculator(league_settings)
        self.stats = self._get_all_stats()
        self.jacobian_matrix = self._calculate_jacobian()

    def _get_all_stats(self):
        """
        Gets a list of all unique stats across all player projections,
        sorted alphabetically for consistency.
        """
        all_stats = set()
        for player in self.players:
            all_stats.update(player.projections.keys())
        return sorted(list(all_stats))

    def _calculate_jacobian(self):
        """
        Calculates the Jacobian matrix.
        Rows: Players
        Columns: Stats
        Value: Partial derivative of fantasy points with respect to the stat
        """
        num_players = len(self.players)
        num_stats = len(self.stats)
        matrix = np.zeros((num_players, num_stats))

        for i, player in enumerate(self.players):
            for j, stat in enumerate(self.stats):
                matrix[i, j] = self._calculate_partial_derivative(player, stat)

        return matrix

In [ ]:
def _calculate_partial_derivative(self, player, stat):
        """
        Calculates the partial derivative of fantasy points with respect to a stat.
        """
        if stat == "passing_yds":
            return self.settings.pass_yd_pts
        elif stat == "passing_tds":
            return self.settings.pass_td_pts
        elif stat == "passing_ints":
            return self.settings.pass_int_pts
        elif stat == "rushing_yds":
            return self.settings.rush_yd_pts
        elif stat == "rushing_tds":
            return self.settings.rush_td_pts
        elif stat == "receptions":
            return self.settings.ppr
        elif stat == "receiving_yds":
            return self.settings.rec_yd_pts
        elif stat == "receiving_tds":
            return self.settings.rec_td_pts
        elif stat == "fumbles_lost":
            return self.settings.fumble_lost_pts
        elif stat == "two_pt_conversions":
            return self.settings.two_pt_conv_pts
        elif stat == "fg_0_39":
            return self.settings.fg_0_39_pts
        elif stat == "fg_40_49":
            return self.settings.fg_40_49_pts
        elif stat == "fg_50_plus":
            return self.settings.fg_50_plus_pts
        elif stat == "fg_missed":
            return self.settings.fg_missed_pts
        elif stat == "xp_made":
            return self.settings.xp_pts
        elif stat == "xp_missed":
            return self.settings.xp_missed_pts
        elif stat == "sacks":
            return self.settings.sack_pts
        elif stat == "interceptions":
            return self.settings.int_pts
        elif stat == "fumbles_recovered":
            return self.settings.fumble_rec_pts
        elif stat == "defensive_tds":
            return self.settings.def_td_pts
        elif stat == "safeties":
            return self.settings.safety_pts
        elif stat == "points_allowed":
            # For points allowed, we approximate the derivative
            points_allowed = player.get_projected_value("points_allowed")
            if points_allowed == 0:
                return 0  # Derivative is 0 at the threshold
            elif 1 <= points_allowed <= 6:
                return (self.settings.pts_allowed_1_6_pts - self.settings.pts_allowed_0_pts) / 6
            elif 7 <= points_allowed <= 13:
                return (self.settings.pts_allowed_7_13_pts - self.settings.pts_allowed_1_6_pts) / 6
            elif 14 <= points_allowed <= 20:
                return (self.settings.pts_allowed_14_20_pts - self.settings.pts_allowed_7_13_pts) / 6
            elif 21 <= points_allowed <= 27:
                 return (self.settings.pts_allowed_21_27_pts - self.settings.pts_allowed_14_20_pts) / 6
            elif 28 <= points_allowed <= 34:
                return (self.settings.pts_allowed_28_34_pts - self.settings.pts_allowed_21_27_pts) / 6
            else:
                return (self.settings.pts_allowed_35_plus_pts - self.settings.pts_allowed_28_34_pts) / 6
        elif stat == "yards_allowed":
            # Similar approximation for yards allowed
            yards_allowed = player.get_projected_value("yards_allowed")
            if yards_allowed < self.settings.yds_allowed_thresholds[0]:
                return 0
            elif yards_allowed < self.settings.yds_allowed_thresholds[1]:
                return (self.settings.yds_allowed_points[1] - self.settings.yds_allowed_points[0]) / (self.settings.yds_allowed_thresholds[1] - self.settings.yds_allowed_thresholds[0])
            elif yards_allowed < self.settings.yds_allowed_thresholds[2]:
                return (self.settings.yds_allowed_points[2] - self.settings.yds_allowed_points[1]) / (self.settings.yds_allowed_thresholds[2] - self.settings.yds_allowed_thresholds[1])
            else:
                return (self.settings.yds_allowed_points[3] - self.settings.yds_allowed_points[2]) / (self.settings.yds_allowed_thresholds[3] - self.settings.yds_allowed_thresholds[2])
        else:
            return 0  # Stat doesn't affect scoring


In [ ]:
def get_jacobian_df(self):
        """
        Returns the Jacobian matrix as a Pandas DataFrame for better visualization.
        """
        player_names = [player.name for player in self.players]
        return pd.DataFrame(self.jacobian_matrix, index=player_names, columns=self.stats)

    def analyze_positional_value(self):
        """
        Analyzes the Jacobian to determine positional value based on stat sensitivities.
        """
        jacobian_df = self.get_jacobian_df()
        positional_summary = {}

        for position in set(p.position for p in self.players):
            pos_df = jacobian_df[jacobian_df.index.isin(
                [p.name for p in self.players if p.position == position]
            )]
            
            # Calculate weighted impact: partial derivative * projected stat value
            weighted_impact = pd.DataFrame(index=pos_df.index)
            for stat in self.stats:
                weighted_impact[stat] = pos_df[stat] * [p.get_projected_value(stat) for p in self.players if p.name in pos_df.index]
                
            positional_summary[position] = {
                "mean_sensitivity": pos_df.abs().mean(),  # Mean absolute sensitivity per stat
                "top_stats": pos_df.abs().mean().nlargest(5).index.tolist(),  # Top 5 most sensitive stats
                "weighted_impact_stats": weighted_impact.abs().mean().nlargest(5).index.tolist(), # Top 5 stats by weighted impact
                "weighted_impact_means": weighted_impact.abs().mean